In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.drop_duplicates(inplace=True)

In [5]:
df.shape

(49582, 2)

# Pre - Processing

In [6]:
df["review"] = df["review"].str.lower()

In [7]:
import re

def remove_url(text):
    return re.sub(r"http\S+", "", text)

df["review"] = df["review"].apply(remove_url)

In [8]:
def remove_puntuations(text):
    return re.sub(r"[^A-Za-z0-9\s]", "", text)

df["review"] = df["review"].apply(remove_puntuations)

In [9]:
def remove_html(text):
    return re.sub(r"<.*>", "", text)

df["review"] = df["review"].apply(remove_html)

In [11]:
!pip install nltk

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------------------------------ --- 1.6/1.7 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 7.9 MB/s eta 0:00:00

   ---------------------------------------- 0/2 [regex]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   ----

In [12]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\JORDAN\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\JORDAN\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\JORDAN\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [13]:
def remove_stop_words(text):
    token = word_tokenize(text)
    stop_words = stopwords.words("english")
    
    for word in token:
        if word in stop_words:
            text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stop_words)
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [14]:
from nltk.stem import PorterStemmer

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    token = word_tokenize(text)
    for word in token:
        stemmed_token = ps.stem(word)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)


df["review"] = df["review"].apply(stemming)

In [15]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y = le.fit_transform(df["sentiment"])

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)
X = tf.fit_transform(df["review"])

In [17]:
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = X_train.toarray()
X_test = X_test.toarray()

# X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
# X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

# y_train_tensor = torch.tensor(y_train, torch.float32)
# y_test_tensor = torch.tensor(y_test, torch.float32)

In [18]:
from torch.utils.data import TensorDataset, DataLoader

train_tensor = TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train).float())
test_tensor = TensorDataset(torch.from_numpy(X_test).float(), torch.from_numpy(y_test).float())

train_loader = DataLoader(train_tensor, batch_size=32, shuffle=True)
tets_loader = DataLoader(test_tensor, batch_size=32, shuffle=True)

In [20]:
import torch.optim as optim


class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layer=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layer = num_layer

        self.rnn = nn.RNN(input_size, hidden_size, num_layer, batch_first=True)

        self.fc_layer = nn.Sequential(
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x):
        # optional
        # Shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layer, x.size(0), self.hidden_size)
        out, _ = self.rnn(x, h0)
        # out => hidden states of the timesteps
        # _(2nd) => final hidden state of last timesteps
        return self.fc_layer(out[:, -1, :])

In [21]:
input_size = X_train.shape[1]

model = RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [25]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()
        Xb = Xb.unsqueeze(1)
        outputs = model(Xb)
        outputs = torch.sigmoid(outputs.squeeze())
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}/{epochs} | Loss = {loss.item()}")

        

Epoch 1/10 | Loss = 0.05069175735116005
Epoch 2/10 | Loss = 0.011375178582966328
Epoch 3/10 | Loss = 0.020676597952842712
Epoch 4/10 | Loss = 0.15851570665836334
Epoch 5/10 | Loss = 0.03012903220951557
Epoch 6/10 | Loss = 0.010201853699982166
Epoch 7/10 | Loss = 0.041967131197452545
Epoch 8/10 | Loss = 0.05282038077712059
Epoch 9/10 | Loss = 0.021787414327263832
Epoch 10/10 | Loss = 0.01151589211076498


In [30]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    total = 0

    for Xb, yb in tets_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        prediction = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
        total += yb.size(0)
        correct_vals += (prediction == yb).sum().item()

In [31]:
print(correct_vals/total, correct_vals, total)

0.8634667742260764 8563 9917
